# 🏆 Train SportsAI LLM with Hugging Face
### Fine-Tuning Open LLMs on the 100-Page Sports Compendium (Cricket, Football, Badminton)

This notebook trains and fine-tunes an open-source LLM (such as `Qwen/Qwen2.5-0.5B-Instruct`, `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, or `meta-llama/Llama-3.2-1B-Instruct`) using **Hugging Face `transformers`**, **`peft` (LoRA/QLoRA)**, and **`trl` (SFTTrainer)**.

## Step 1: Install Hugging Face Libraries
Run this in Google Colab (T4 GPU recommended) or your local environment:

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes huggingface_hub

## Step 2: Load Model and Tokenizer
Select any base model from Hugging Face Hub:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # or "TinyLlama/TinyLlama-1.1B-Chat-v1.0" or "meta-llama/Llama-3.2-1B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Optional 4-bit quantization for GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=torch.cuda.is_available(),
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
) if torch.cuda.is_available() else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

# Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 3: Load the 100-Page Sports Compendium Dataset

In [ ]:
from datasets import load_dataset

# Load local jsonl dataset generated by prepare_dataset.py
dataset = load_dataset("json", data_files={
    "train": "dataset/sports_train_chat.jsonl",
    "validation": "dataset/sports_val_chat.jsonl"
})

print("Sample training example:")
print(dataset["train"][0])

## Step 4: Fine-Tune with SFTTrainer / Hugging Face Trainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sportsai-hf-model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_ratio=0.05,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args
)

print("Starting Hugging Face Training...")
trainer.train()
print("Training Complete!")

## Step 5: Test Model Inference & Save to Hugging Face Hub

In [ ]:
from transformers import pipeline

# Test inference
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
prompt = "Explain the 3 pillars of LBW in cricket according to MCC Law 36."

messages = [
    {"role": "system", "content": "You are SportsAI, an authoritative zero-hallucination sports engine for Cricket, Football, and Badminton."},
    {"role": "user", "content": prompt}
]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
result = pipe(formatted_prompt, max_new_tokens=256, temperature=0.2)
print("\n--- AI Output ---\n")
print(result[0]["generated_text"])

# Save model locally
trainer.save_model("./final_sportsai_llm")
tokenizer.save_pretrained("./final_sportsai_llm")
print("\nModel successfully saved to ./final_sportsai_llm!")

# Optional: Push to Hugging Face Hub
# from huggingface_hub import login
# login() # Enter your Hugging Face Token
# model.push_to_hub("your-hf-username/SportsAI-TriSport-LLM")
# tokenizer.push_to_hub("your-hf-username/SportsAI-TriSport-LLM")